# ACE ML Models - Model Registry\n
\n
This notebook trains ML models for the ACE Intelligence Agent:\n
- **Service Request Volume Forecasting** - Predict future monthly service request volume\n
- **Member Churn Prediction** - Classify members at risk of non-renewal or cancellation\n
- **Response Success Prediction** - Predict service fulfillment success based on conditions\n
\n
All models are registered to Snowflake Model Registry and can be added as tools to the Intelligence Agent.\n
\n
## Prerequisites\n
\n
**Required Packages** (configured automatically):\n
- `snowflake-ml-python`\n
- `scikit-learn`\n
- `xgboost`\n
- `matplotlib`\n
\n
**Database Context:**\n
- **Database:** AAA_INTELLIGENCE  \n
- **Schema:** ANALYTICS  \n
- **Warehouse:** AAA_WH\n
\n
**Note:** This notebook uses Snowflake Model Registry. Ensure you have appropriate permissions to create and register models.

## Import Required Packages

In [ ]:
# Import Python packages\n
import pandas as pd\n
import warnings\n
warnings.filterwarnings('ignore')\n
\n
# Import Snowpark\n
from snowflake.snowpark.context import get_active_session\n
import snowflake.snowpark.functions as F\n
import snowflake.snowpark.types as T\n
from snowflake.snowpark import Window\n
\n
# Import Snowpark ML\n
from snowflake.ml.modeling.preprocessing import StandardScaler, OneHotEncoder\n
from snowflake.ml.modeling.pipeline import Pipeline\n
from snowflake.ml.modeling.linear_model import LinearRegression, LogisticRegression\n
from snowflake.ml.modeling.ensemble import RandomForestClassifier\n
from snowflake.ml.modeling.metrics import mean_squared_error, mean_absolute_error, accuracy_score, roc_auc_score\n
from snowflake.ml.registry import Registry\n
\n
print(\"✅ Packages imported successfully\")

## Connect to Snowflake\n
\n
Get active session and set context to ACE database.

In [ ]:
# Get active Snowflake session\n
session = get_active_session()\n
\n
# Set context\n
session.use_database('AAA_INTELLIGENCE')\n
session.use_schema('ANALYTICS')\n
session.use_warehouse('AAA_WH')\n
\n
print(f\"✅ Connected - Role: {session.get_current_role()}\")\n
print(f\"   Warehouse: {session.get_current_warehouse()}\")\n
print(f\"   Database.Schema: {session.get_fully_qualified_current_schema()}\")

---\n
# MODEL 1: Service Request Volume Forecasting\n
\n
Predict future monthly service request volume based on historical patterns, seasonality, and service types.

### Prepare Service Volume Training Data

In [ ]:
# Get monthly service request volume data with features\n
service_volume_df = session.sql(\"\"\"\n
SELECT\n
    DATE_TRUNC('month', request_timestamp)::DATE AS service_month,\n
    MONTH(request_timestamp) AS month_num,\n
    YEAR(request_timestamp) AS year_num,\n
    COUNT(DISTINCT service_id)::FLOAT AS total_service_requests,\n
    COUNT(DISTINCT member_id)::FLOAT AS unique_members,\n
    COUNT(DISTINCT vehicle_id)::FLOAT AS unique_vehicles,\n
    AVG(CASE WHEN priority = 'HIGH' THEN 1.0 ELSE 0.0 END)::FLOAT AS high_priority_ratio,\n
    COUNT(DISTINCT CASE WHEN service_type = 'TOWING' THEN service_id END)::FLOAT AS towing_count,\n
    COUNT(DISTINCT CASE WHEN service_type = 'TIRE_CHANGE' THEN service_id END)::FLOAT AS tire_count,\n
    COUNT(DISTINCT CASE WHEN service_type = 'BATTERY_JUMP' THEN service_id END)::FLOAT AS battery_count,\n
    AVG(CASE WHEN weather_condition IN ('RAIN', 'SNOW', 'ICE') THEN 1.0 ELSE 0.0 END)::FLOAT AS bad_weather_ratio\n
FROM RAW.SERVICE_REQUESTS\n
WHERE request_timestamp >= DATEADD('month', -36, CURRENT_DATE())\n
  AND request_timestamp < CURRENT_DATE()\n
GROUP BY DATE_TRUNC('month', request_timestamp), MONTH(request_timestamp), YEAR(request_timestamp)\n
ORDER BY service_month\n
\"\"\")\n
\n
print(f\"Service volume data: {service_volume_df.count()} months\")\n
service_volume_df.show(5)

### Split Data and Train Service Volume Model

In [ ]:
# Train/test split (last 6 months for testing)\n
train_service_volume = service_volume_df.filter(F.col(\"SERVICE_MONTH\") < F.dateadd(\"month\", F.lit(-6), F.current_date()))\n
test_service_volume = service_volume_df.filter(F.col(\"SERVICE_MONTH\") >= F.dateadd(\"month\", F.lit(-6), F.current_date()))\n
\n
# Drop SERVICE_MONTH (DATE type not supported in pipeline)\n
train_service_volume = train_service_volume.drop(\"SERVICE_MONTH\")\n
test_service_volume = test_service_volume.drop(\"SERVICE_MONTH\")\n
\n
# Create pipeline with scaling and regression\n
service_volume_pipeline = Pipeline([\n
    (\"Scaler\", StandardScaler(\n
        input_cols=[\"MONTH_NUM\", \"UNIQUE_MEMBERS\", \"UNIQUE_VEHICLES\", \"HIGH_PRIORITY_RATIO\", \n
                   \"TOWING_COUNT\", \"TIRE_COUNT\", \"BATTERY_COUNT\", \"BAD_WEATHER_RATIO\"],\n
        output_cols=[\"MONTH_NUM_SCALED\", \"UNIQUE_MEMBERS_SCALED\", \"UNIQUE_VEHICLES_SCALED\", \n
                    \"HIGH_PRIORITY_RATIO_SCALED\", \"TOWING_COUNT_SCALED\", \"TIRE_COUNT_SCALED\", \n
                    \"BATTERY_COUNT_SCALED\", \"BAD_WEATHER_RATIO_SCALED\"]\n
    )),\n
    (\"LinearRegression\", LinearRegression(\n
        label_cols=[\"TOTAL_SERVICE_REQUESTS\"],\n
        output_cols=[\"PREDICTED_SERVICE_REQUESTS\"]\n
    ))\n
])\n
\n
# Train model\n
service_volume_pipeline.fit(train_service_volume)\n
print(\"✅ Service request volume forecasting model trained\")

### Evaluate and Register Service Volume Model

In [ ]:
# Make predictions on test set\n
test_predictions = service_volume_pipeline.predict(test_service_volume)\n
\n
# Calculate metrics\n
mae = mean_absolute_error(df=test_predictions, y_true_col_names=\"TOTAL_SERVICE_REQUESTS\", y_pred_col_names=\"PREDICTED_SERVICE_REQUESTS\")\n
mse = mean_squared_error(df=test_predictions, y_true_col_names=\"TOTAL_SERVICE_REQUESTS\", y_pred_col_names=\"PREDICTED_SERVICE_REQUESTS\")\n
rmse = mse ** 0.5\n
\n
metrics = {\"mae\": round(mae, 2), \"rmse\": round(rmse, 2)}\n
print(f\"Model metrics: {metrics}\")\n
\n
# Register model\n
reg = Registry(session)\n
reg.log_model(\n
    model=service_volume_pipeline,\n
    model_name=\"SERVICE_VOLUME_PREDICTOR\",\n
    version_name=\"V1\",\n
    comment=\"Predicts monthly service request volume based on historical patterns, service mix, and weather conditions using Linear Regression\",\n
    metrics=metrics\n
)\n
\n
print(\"✅ Service volume model registered to Model Registry as SERVICE_VOLUME_PREDICTOR\")

---\n
# MODEL 2: Member Churn Prediction\n
\n
Classify members as likely to churn (non-renewal or cancellation) based on service usage patterns and satisfaction.

### Prepare Churn Training Data

In [ ]:
# Get member features for churn prediction\n
churn_df = session.sql(\"\"\"\n
SELECT\n
    m.member_id,\n
    m.membership_level,\n
    m.risk_score::FLOAT AS risk_score,\n
    m.lifetime_value::FLOAT AS lifetime_value,\n
    DATEDIFF('day', m.membership_start_date, CURRENT_DATE())::FLOAT AS membership_days,\n
    DATEDIFF('day', CURRENT_DATE(), m.membership_renewal_date)::FLOAT AS days_to_renewal,\n
    m.is_auto_renew::BOOLEAN AS is_auto_renew,\n
    -- Service usage patterns (last 6 months)\n
    COUNT(DISTINCT sr.service_id)::FLOAT AS service_requests_6m,\n
    COUNT(DISTINCT CASE WHEN sr.service_type = 'TOWING' THEN sr.service_id END)::FLOAT AS towing_requests_6m,\n
    -- Service fulfillment metrics\n
    AVG(sf.response_time_minutes)::FLOAT AS avg_response_time,\n
    AVG(sf.member_satisfaction_score)::FLOAT AS avg_satisfaction_score,\n
    COUNT(DISTINCT CASE WHEN sf.member_satisfaction_score <= 2 THEN sf.service_id END)::FLOAT AS low_satisfaction_count,\n
    -- Transaction history\n
    COUNT(DISTINCT mt.transaction_id)::FLOAT AS total_transactions,\n
    SUM(CASE WHEN mt.transaction_type = 'RENEWAL' THEN 1 ELSE 0 END)::FLOAT AS renewal_count,\n
    -- Predictive scores\n
    AVG(ps.churn_risk_score)::FLOAT AS avg_churn_risk_score,\n
    -- Target: Is churned (membership status)\n
    (m.membership_status = 'CANCELLED')::BOOLEAN AS is_churned\n
FROM RAW.MEMBERS m\n
LEFT JOIN RAW.SERVICE_REQUESTS sr ON m.member_id = sr.member_id \n
    AND sr.request_timestamp >= DATEADD('month', -6, CURRENT_DATE())\n
LEFT JOIN RAW.SERVICE_FULFILLMENT sf ON sr.service_id = sf.service_id\n
LEFT JOIN RAW.MEMBER_TRANSACTIONS mt ON m.member_id = mt.member_id\n
LEFT JOIN RAW.PREDICTIVE_SCORES ps ON m.member_id = ps.member_id\n
WHERE m.membership_status IN ('ACTIVE', 'CANCELLED')\n
  AND m.membership_start_date <= DATEADD('month', -12, CURRENT_DATE()) -- At least 1 year old\n
GROUP BY m.member_id, m.membership_level, m.risk_score, m.lifetime_value, \n
         m.membership_start_date, m.membership_renewal_date, m.is_auto_renew, m.membership_status\n
HAVING COUNT(DISTINCT sr.service_id) > 0 OR COUNT(DISTINCT mt.transaction_id) > 0\n
LIMIT 10000  -- Limit for faster training\n
\"\"\")\n
\n
print(f\"Churn data: {churn_df.count()} members\")\n
churn_df.show(5)

### Train Churn Classification Model

In [ ]:
# Train/test split (80/20)\n
train_churn, test_churn = churn_df.random_split([0.8, 0.2], seed=42)\n
\n
# Drop MEMBER_ID\n
train_churn = train_churn.drop(\"MEMBER_ID\")\n
test_churn = test_churn.drop(\"MEMBER_ID\")\n
\n
# Create pipeline with preprocessing and classification\n
churn_pipeline = Pipeline([\n
    (\"Encoder\", OneHotEncoder(\n
        input_cols=[\"MEMBERSHIP_LEVEL\"],\n
        output_cols=[\"MEMBERSHIP_LEVEL_ENCODED\"],\n
        drop_input_cols=True,  # Drop original string columns after encoding\n
        handle_unknown=\"ignore\"\n
    )),\n
    (\"Classifier\", RandomForestClassifier(\n
        label_cols=[\"IS_CHURNED\"],\n
        output_cols=[\"CHURN_PREDICTION\"],\n
        n_estimators=100,\n
        max_depth=10,\n
        random_state=42\n
    ))\n
])\n
\n
# Train model\n
churn_pipeline.fit(train_churn)\n
print(\"✅ Member churn classification model trained\")

### Evaluate and Register Churn Model

In [ ]:
# Make predictions\n
churn_predictions = churn_pipeline.predict(test_churn)\n
\n
# Calculate metrics\n
accuracy = accuracy_score(df=churn_predictions, y_true_col_names=\"IS_CHURNED\", y_pred_col_names=\"CHURN_PREDICTION\")\n
churn_metrics = {\"accuracy\": round(accuracy, 4)}\n
print(f\"Churn model metrics: {churn_metrics}\")\n
\n
# Register model\n
reg.log_model(\n
    model=churn_pipeline,\n
    model_name=\"MEMBER_CHURN_PREDICTOR\",\n
    version_name=\"V1\",\n
    comment=\"Predicts member churn probability using Random Forest based on service usage patterns and satisfaction metrics\",\n
    metrics=churn_metrics\n
)\n
\n
print(\"✅ Churn model registered to Model Registry as MEMBER_CHURN_PREDICTOR\")

---\n
# MODEL 3: Response Success Prediction\n
\n
Predict which service requests are likely to be completed successfully within SLA based on conditions and resources.

### Prepare Response Success Data

In [ ]:
# Get service request features for success prediction\n
response_success_df = session.sql(\"\"\"\n
SELECT\n
    sr.service_id,\n
    sr.service_type,\n
    sr.service_category,\n
    sr.priority,\n
    sr.location_type,\n
    sr.weather_condition,\n
    sr.temperature_f::FLOAT AS temperature_f,\n
    sr.channel,\n
    -- Time features\n
    HOUR(sr.request_timestamp)::INT AS request_hour,\n
    DAYOFWEEK(sr.request_timestamp)::INT AS request_dow,\n
    -- Regional features\n
    reg.average_response_time_minutes::FLOAT AS region_avg_response,\n
    reg.active_technicians::FLOAT AS region_technicians,\n
    reg.active_trucks::FLOAT AS region_trucks,\n
    -- Member features\n
    m.membership_level,\n
    v.vehicle_type,\n
    -- Technician features\n
    t.certification_level,\n
    t.average_response_time_minutes::FLOAT AS tech_avg_response,\n
    -- Success criteria: Completed within regional SLA and high satisfaction\n
    (sf.service_outcome = 'COMPLETED' \n
     AND sf.response_time_minutes <= reg.target_response_time_minutes\n
     AND (sf.member_satisfaction_score >= 4 OR sf.member_satisfaction_score IS NULL))::BOOLEAN AS response_successful\n
FROM RAW.SERVICE_REQUESTS sr\n
JOIN RAW.SERVICE_FULFILLMENT sf ON sr.service_id = sf.service_id\n
JOIN RAW.SERVICE_TECHNICIANS t ON sf.technician_id = t.technician_id\n
JOIN RAW.MEMBERS m ON sr.member_id = m.member_id\n
LEFT JOIN RAW.VEHICLES v ON sr.vehicle_id = v.vehicle_id\n
LEFT JOIN RAW.SERVICE_REGIONS reg ON t.service_region = reg.region_name\n
WHERE sr.request_timestamp >= DATEADD('month', -12, CURRENT_DATE())\n
  AND sf.completion_timestamp IS NOT NULL\n
\"\"\")\n
\n
print(f\"Response success data: {response_success_df.count()} service requests\")\n
response_success_df.show(5)

### Train Response Success Model

In [ ]:
# Split data\n
train_response, test_response = response_success_df.random_split([0.8, 0.2], seed=42)\n
\n
# Drop SERVICE_ID\n
train_response = train_response.drop(\"SERVICE_ID\")\n
test_response = test_response.drop(\"SERVICE_ID\")\n
\n
# Create pipeline\n
response_pipeline = Pipeline([\n
    (\"Encoder\", OneHotEncoder(\n
        input_cols=[\"SERVICE_TYPE\", \"SERVICE_CATEGORY\", \"PRIORITY\", \"LOCATION_TYPE\", \n
                   \"WEATHER_CONDITION\", \"CHANNEL\", \"MEMBERSHIP_LEVEL\", \"VEHICLE_TYPE\", \n
                   \"CERTIFICATION_LEVEL\"],\n
        output_cols=[\"SERVICE_TYPE_ENC\", \"SERVICE_CATEGORY_ENC\", \"PRIORITY_ENC\", \"LOCATION_TYPE_ENC\",\n
                    \"WEATHER_CONDITION_ENC\", \"CHANNEL_ENC\", \"MEMBERSHIP_LEVEL_ENC\", \"VEHICLE_TYPE_ENC\",\n
                    \"CERTIFICATION_LEVEL_ENC\"],\n
        drop_input_cols=True,\n
        handle_unknown=\"ignore\"\n
    )),\n
    (\"Classifier\", LogisticRegression(\n
        label_cols=[\"RESPONSE_SUCCESSFUL\"],\n
        output_cols=[\"SUCCESS_PREDICTION\"]\n
    ))\n
])\n
\n
# Train\n
response_pipeline.fit(train_response)\n
print(\"✅ Response success model trained\")

### Evaluate and Register Response Success Model

In [ ]:
# Predict on test set\n
response_predictions = response_pipeline.predict(test_response)\n
\n
# Calculate accuracy\n
response_accuracy = accuracy_score(df=response_predictions, \n
                                 y_true_col_names=\"RESPONSE_SUCCESSFUL\",\n
                                 y_pred_col_names=\"SUCCESS_PREDICTION\")\n
response_metrics = {\"accuracy\": round(response_accuracy, 4)}\n
print(f\"Response success model metrics: {response_metrics}\")\n
\n
# Register model\n
reg.log_model(\n
    model=response_pipeline,\n
    model_name=\"RESPONSE_SUCCESS_PREDICTOR\",\n
    version_name=\"V1\",\n
    comment=\"Predicts roadside service success (completed within SLA with high satisfaction) using Logistic Regression based on conditions, resources, and technician skills\",\n
    metrics=response_metrics\n
)\n
\n
print(\"✅ Response success model registered to Model Registry as RESPONSE_SUCCESS_PREDICTOR\")

---\n
# Verify Models in Registry

In [ ]:
# Show all models in the registry\n
print(\"Models in registry:\")\n
reg.show_models()\n
\n
# Show versions for service volume model\n
print(\"\\nService Volume Predictor versions:\")\n
reg.get_model(\"SERVICE_VOLUME_PREDICTOR\").show_versions()\n
\n
# Show versions for churn model  \n
print(\"\\nMember Churn Predictor versions:\")\n
reg.get_model(\"MEMBER_CHURN_PREDICTOR\").show_versions()\n
\n
# Show versions for response success model\n
print(\"\\nResponse Success Predictor versions:\")\n
reg.get_model(\"RESPONSE_SUCCESS_PREDICTOR\").show_versions()\n
\n
print(\"\\n✅ All models registered and ready to add to Intelligence Agent\")

---\n
# Test Model Inference\n
\n
Test calling each model to make predictions.

In [ ]:
# Test service volume forecast on recent data\n
service_model = reg.get_model(\"SERVICE_VOLUME_PREDICTOR\").default\n
recent_service = service_volume_df.limit(3).drop(\"SERVICE_MONTH\")\n
service_preds = service_model.run(recent_service, function_name=\"predict\")\n
print(\"Service Volume predictions:\")\n
service_preds.select(\"TOTAL_SERVICE_REQUESTS\", \"PREDICTED_SERVICE_REQUESTS\").show()\n
\n
# Test churn prediction on sample members\n
churn_model = reg.get_model(\"MEMBER_CHURN_PREDICTOR\").default\n
sample_members = churn_df.limit(5).drop(\"MEMBER_ID\")\n
churn_preds = churn_model.run(sample_members, function_name=\"predict\")\n
print(\"\\nChurn predictions:\")\n
churn_preds.select(\"IS_CHURNED\", \"CHURN_PREDICTION\").show()\n
\n
# Test response success prediction\n
response_model = reg.get_model(\"RESPONSE_SUCCESS_PREDICTOR\").default\n
sample_responses = response_success_df.limit(5).drop(\"SERVICE_ID\")\n
response_preds = response_model.run(sample_responses, function_name=\"predict\")\n
print(\"\\nResponse Success predictions:\")\n
response_preds.select(\"RESPONSE_SUCCESSFUL\", \"SUCCESS_PREDICTION\").show()\n
\n
print(\"\\n✅ All models tested successfully!\")

---\n
# Next Steps\n
\n
## Add Models to Intelligence Agent\n
\n
**Option 1: Using the SQL Script (Easiest)**\n
Run `sql/agent/08_create_intelligence_agent.sql` which automatically configures all 3 ML models.\n
\n
**Option 2: Manual Configuration in Snowsight**\n
1. In Snowsight → AI & ML → Agents → AAA_INTELLIGENCE_AGENT\n
2. Go to Tools → + Add → Function\n
3. Add each model wrapper procedure:\n
   - **PREDICT_SERVICE_VOLUME** (from `sql/ml/07_create_model_wrapper_functions.sql`)\n
   - **PREDICT_MEMBER_CHURN** (from `sql/ml/07_create_model_wrapper_functions.sql`)\n
   - **PREDICT_RESPONSE_SUCCESS** (from `sql/ml/07_create_model_wrapper_functions.sql`)\n
\n
## Example Questions for Agent\n
\n
- \"Predict service request volume for the next 6 months\"\n
- \"Which members are at high risk of cancellation?\"\n
- \"What is the predicted success rate for a towing request on Highway 101 during rain?\"\n
- \"Forecast demand for battery jump services next quarter\"\n
\n
The models will now be available as tools your agent can use!